In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [3]:
df = pd.read_csv('../datasets/covid_toy.csv')

In [16]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [19]:
df.sample(5)

,age,gender,fever,cough,city,has_covid
83,17,Female,104.0,Mild,Kolkata,No
65,69,Female,102.0,Mild,Bangalore,No
62,56,Female,104.0,Strong,Bangalore,Yes
51,11,Female,100.0,Strong,Kolkata,Yes
55,81,Female,101.0,Mild,Mumbai,Yes


In [5]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(df.drop(columns='has_covid'),df['has_covid'],test_size=0.3,random_state=42)

In [10]:
X_train

,age,gender,fever,cough,city
11,65,Female,98.0,Mild,Mumbai
47,18,Female,104.0,Mild,Bangalore
85,16,Female,103.0,Mild,Bangalore
28,16,Male,104.0,Mild,Kolkata
93,27,Male,100.0,Mild,Kolkata
...,...,...,...,...,...
60,24,Female,102.0,Strong,Bangalore
71,75,Female,104.0,Strong,Delhi
14,51,Male,104.0,Mild,Bangalore
92,82,Female,102.0,Strong,Kolkata


In [12]:
si = SimpleImputer()
#To find the mean of fever we will ue fit and to fill the missing value transform, both at same time using fit transform
X_train_fever = si.fit_transform(X_train[['fever']])
X_test_fever = si.fit_transform(X_test[['fever']])

In [14]:
X_train_fever.shape

(70, 1)

In [35]:
oe = OrdinalEncoder(categories=[['Mild','Strong']])

X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.fit_transform(X_test[['cough']])

In [36]:
X_train_cough.shape

(70, 1)

In [30]:
oh = OneHotEncoder(drop='first',sparse_output=False)

X_train_gender_city = oh.fit_transform(X_train[['gender','city']])
X_test_gender_city = oh.fit_transform(X_test[['gender','city']])

In [33]:
X_train_gender_city.shape

(70, 4)

In [37]:
X_train_age = X_train.drop(columns=['gender','city','fever','cough'])
X_test_age = X_test.drop(columns=['gender','city','fever','cough'])

In [39]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

In [41]:
X_train_transformed.shape

(70, 7)

# Now Using Column Transformer

In [43]:
# List of (name, transformer, columns) to pass in column transformer

In [44]:
from sklearn.compose import ColumnTransformer

In [49]:
transformer = ColumnTransformer(transformers=[('tnf1',SimpleImputer(),['fever']),
                                 ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
                                 ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])],remainder='passthrough')

In [51]:
transformer.fit_transform(X_train).shape

(70, 7)

In [52]:
transformer.transform(X_test).shape

(30, 7)